# HealthConnect Rwanda — Week 6 Machine Learning Engineering

**Project:** AI-Powered Clinical Risk Prediction and Healthcare Intelligence System  
**Purpose:** Consolidate existing data-readiness findings, model evaluation, patient and facility clustering, explainability, anomaly review, and documented limitations.

> **Important:** This is exploratory educational analysis. The baseline models are weak and are **not suitable for clinical, insurance, or billing decisions**. Missing target labels and relationships are not fabricated.

## Scope covered
- Review prepared modelling datasets and existing baseline metrics.
- Summarize classification and regression performance.
- Review patient and facility clustering outputs.
- Review available feature-importance and anomaly reports.
- Record target and data-quality limitations.
- State next steps for responsible use.

The modelling and reporting scripts remain the source of results; this notebook consolidates their outputs rather than silently retraining models.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
# If the notebook is opened from the Notebook folder, use its parent as project root.
if not (ROOT / "Reports").exists() and (ROOT.parent / "Reports").exists():
    ROOT = ROOT.parent

REPORTS = ROOT / "Reports"
PROCESSED = ROOT / "Data" / "Processed_Datasets"
MODELS = ROOT / "Models"

print("Project root:", ROOT)
print("Reports folder exists:", REPORTS.exists())
print("Processed datasets folder exists:", PROCESSED.exists())
print("Models folder exists:", MODELS.exists())

Project root: c:\Users\USER\Documents\HealthConnect-Rwanda-ML-Engineering
Reports folder exists: True
Processed datasets folder exists: True
Models folder exists: True


## 2. Prepared datasets

In [2]:
prepared_files = [
    "patient_satisfaction_model.csv",
    "high_risk_severity_proxy.csv",
    "billing_amount_model.csv",
    "length_of_stay_model.csv",
    "claims_approved_pending_alternative.csv",
    "patient_clustering.csv",
    "week6_target_limitations.csv",
]

dataset_summary = []
for filename in prepared_files:
    path = PROCESSED / filename
    if path.exists():
        df = pd.read_csv(path)
        dataset_summary.append({
            "file": filename,
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": ", ".join(df.columns.astype(str))
        })
    else:
        dataset_summary.append({
            "file": filename, "rows": None, "columns": None,
            "column_names": "File not found"
        })

display(pd.DataFrame(dataset_summary))

,file,rows,columns,column_names
0,patient_satisfaction_model.csv,1756,19,"AppointmentID, SatisfactionClass, PatientID, D..."
1,high_risk_severity_proxy.csv,5000,19,"AppointmentID, HighRiskProxy, PatientID, Docto..."
2,billing_amount_model.csv,5000,19,"AppointmentID, Amount, PatientID, DoctorID, Fa..."
3,length_of_stay_model.csv,586,19,"AppointmentID, LengthOfStayDays, PatientID, Do..."
4,claims_approved_pending_alternative.csv,42036,5,"Provider, ClaimAmount, Amount, AppointmentID, ..."
5,patient_clustering.csv,20000,12,"PatientID, Age, Gender, MaritalStatus, Occupat..."
6,week6_target_limitations.csv,4,3,"Task, Status, Limitation"


## 3. Baseline model metrics

This section reads the existing baseline metrics CSV. It does not retrain models. Values should be interpreted alongside the documented label limitations and the weakness of the baselines.

In [3]:
metrics_path = REPORTS / "week6_baseline_metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    print(f"Baseline metrics file not found: {metrics_path}")

,task,rows,test_rows,selected_model,validation_f1_macro,accuracy,precision_macro,recall_macro,f1_macro,baseline_accuracy,...,roc_auc,validation_MAE,MAE,RMSE,R2,MAPE_percent,baseline_MAE,baseline_RMSE,baseline_R2,clusters
0,Patient satisfaction classification (rating-de...,1756,352.0,Random Forest,0.390931,0.377841,0.367880,0.361467,0.363515,0.409091,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,High-risk patient classification (severity proxy),5000,1000.0,Decision Tree,0.482976,0.502000,0.482791,0.480587,0.474184,0.676000,...,0.474706,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Insurance claim outcome (Approved vs Pending a...,42036,8408.0,Random Forest,0.501718,0.500238,0.500238,0.500238,0.500238,0.500238,...,0.499894,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Billing amount prediction,5000,1000.0,Dummy (mean),NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,71708.842758,70207.143520,81883.154322,-0.002288,113.970188,70207.143520,81883.154322,-0.002288,NaN
4,Length of stay prediction,586,118.0,Extra Trees,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.266446,2.474539,3.012954,-0.155440,93.635245,2.353325,2.816652,-0.009785,NaN
5,Patient clustering,20000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
6,Facility clustering,20,NaN,KMeans (3 clusters),NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0


### Interpretation of the current recorded baseline results

- **Patient satisfaction:** accuracy 0.3778; macro F1 0.3635; macro one-vs-rest ROC-AUC 0.4991. Performance is weak.
- **High-risk classification:** accuracy 0.5020; macro F1 0.4742; ROC-AUC 0.4747. The target is a severity-derived proxy, not a clinically validated risk label.
- **Claims:** Approved-versus-Pending alternative accuracy 0.5002 and ROC-AUC 0.4999, approximately chance-level. The source contains no Rejected claims, so the required three-class task cannot be trained as specified.
- **Billing amount:** MAE RWF 70,207.14; RMSE RWF 81,883.15; R² −0.0023; MAPE 113.97%. The model does not outperform the mean baseline on these reported regression measures.
- **Length of stay:** MAE 2.47 days; RMSE 3.01 days; R² −0.1554; MAPE 93.64%. Negative R² indicates worse performance than predicting the test-set mean.
- **Patient clustering:** four exploratory clusters across 20,000 patients.
- **Facility clustering:** three exploratory clusters across 20 facilities, using direct facility-level record counts. Resource counts are mostly similar, so the clusters are driven mainly by appointment-record volume.

Metrics are read from the current `Reports/week6_baseline_metrics.csv`; interpret them with the documented target limitations.


## 4. Model comparison and evaluation report

In [4]:
comparison_path = REPORTS / "Week6_Model_Comparison.csv"
evaluation_path = REPORTS / "Week6_Model_Evaluation_Report.txt"

if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print("Model comparison CSV not found:", comparison_path)

if evaluation_path.exists():
    print(evaluation_path.read_text(encoding="utf-8", errors="replace"))
else:
    print("Evaluation report not found:", evaluation_path)

,task,rows,test_rows,selected_model,validation_f1_macro,accuracy,precision_macro,recall_macro,f1_macro,baseline_accuracy,...,R2,MAPE_percent,baseline_MAE,baseline_RMSE,baseline_R2,clusters,business_problem,strengths,limitations,recommended_use_case
0,Patient satisfaction classification (rating-de...,1756,352.0,Random Forest,0.390931,0.377841,0.367880,0.361467,0.363515,0.409091,...,NaN,NaN,NaN,NaN,NaN,NaN,Understand factors associated with patient sat...,Provides a baseline for exploring rating-deriv...,Test accuracy is 37.78%; labels are rating-der...,Exploratory analysis only; not a reliable sati...
1,High-risk patient classification (severity proxy),5000,1000.0,Decision Tree,0.482976,0.502000,0.482791,0.480587,0.474184,0.676000,...,NaN,NaN,NaN,NaN,NaN,NaN,Explore patient risk stratification,Provides a baseline for examining associations...,Test accuracy is 50.20%; labels are not clinic...,Research and feature exploration only; not cli...
2,Insurance claim outcome (Approved vs Pending a...,42036,8408.0,Random Forest,0.501718,0.500238,0.500238,0.500238,0.500238,0.500238,...,NaN,NaN,NaN,NaN,NaN,NaN,Explore automated claims outcome classification,Establishes an Approved-versus-Pending baselin...,Performance is near chance; no Rejected exampl...,Exploratory analysis only until representative...
3,Billing amount prediction,5000,1000.0,Dummy (mean),NaN,NaN,NaN,NaN,NaN,NaN,...,-0.002288,113.970188,70207.143520,81883.154322,-0.002288,NaN,Estimate healthcare billing amounts,Provides a measurable regression baseline,"MAE is approximately RWF 70,207 and R² is -0.0...",Baseline comparison only; not operational bill...
4,Length of stay prediction,586,118.0,Extra Trees,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.155440,93.635245,2.353325,2.816652,-0.009785,NaN,Estimate hospital length of stay,Provides a baseline for evaluating length-of-s...,MAE is approximately 2.47 days and R² is -0.15...,Further research after improving the data and ...
5,Patient clustering,20000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.0,Explore groups of patients with similar record...,Identifies four exploratory patient clusters,Clusters are not clinical diagnoses or validat...,Exploratory segmentation for further analysis
6,Facility clustering,20,NaN,KMeans (3 clusters),NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,Explore differences in recorded facility activ...,Groups 20 facilities into three exploratory cl...,"Resource counts are very similar, so appointme...",Preliminary exploration only; not facility ran...


HealthConnect Rwanda Week 6 - Baseline Model Evaluation

This report summarizes existing baseline metrics; it does not retrain models.
Results are exploratory and should not be interpreted as clinically validated.

## Results

### Patient satisfaction classification (rating-derived)
- rows: 1756
- test_rows: 352.0000
- selected_model: Random Forest
- validation_f1_macro: 0.3909
- accuracy: 0.3778
- precision_macro: 0.3679
- recall_macro: 0.3615
- f1_macro: 0.3635
- baseline_accuracy: 0.4091
- baseline_f1_macro: 0.1935
- roc_auc_ovr_macro: 0.4991
- note: Dataset labels are proxies/observed alternatives; not clinical validation.
- roc_auc: not reported
- validation_MAE: not reported
- MAE: not reported
- RMSE: not reported
- R2: not reported
- MAPE_percent: not reported
- baseline_MAE: not reported
- baseline_RMSE: not reported
- baseline_R2: not reported
- clusters: not reported

### High-risk patient classification (severity proxy)
- rows: 5000
- test_rows: 1,000.00
- selected_model: D

## 5. Explainability review

Permutation feature importance is summarized in the report below. The project also includes SHAP explanations generated by `week6_shap_explainability.py`; the SHAP report and plot are saved in `Reports/`.


In [5]:
explainability_path = REPORTS / "Week6_Explainability_Report.txt"
if explainability_path.exists():
    print(explainability_path.read_text(encoding="utf-8", errors="replace"))
else:
    print("Explainability report not found:", explainability_path)

importance_files = sorted(REPORTS.glob("feature_importance_*.csv"))
print(f"\nFeature-importance CSV files found: {len(importance_files)}")
for path in importance_files:
    print("-", path.name)

## Responsible AI and Interpretation

### Interpretation Limitations
Permutation importance measures how much shuffling a feature affects the fitted model's score. It does not establish causation or prove that a feature is clinically meaningful. Because the baseline models perform poorly, the rankings should be treated as exploratory rather than reliable clinical explanations.

### Key Observations
- **Patient satisfaction:** The model assigns its highest reported importance to HeightCm, Date_month, and MaritalStatus. The model performs poorly, so these rankings should not be interpreted as dependable drivers of patient satisfaction.
- **High-risk patient classification:** HeightCm and BMI have the highest reported importance. The target is a severity proxy, and the model performs worse than the majority-class baseline. These features should not be treated as validated clinical risk factors.
- **Insurance claims:** ClaimAmount, Amount, and Provider have the highest reported importance.

### Explainability limitations

Permutation importance reflects how a feature affects a fitted model's score when shuffled; it does not establish causation. SHAP outputs are also available for a separately fitted decision-tree model on the `HighRiskProxy` target. See `Reports/Week6_SHAP_Explainability_Report.txt` and `Reports/Week6_SHAP_Feature_Importance.png`. Classification baselines are weak. Billing feature importance is marked `uninformative_zero_importance` because shuffling sampled features did not measurably change the score. Length-of-stay importance is exploratory because the regression model performs worse than the mean baseline. These rankings explain model behaviour and are not clinical findings. LIME is not implemented.

## 6. Anomaly review

In [6]:
anomaly_path = REPORTS / "Week6_Anomaly_Review.txt"
if anomaly_path.exists():
    print(anomaly_path.read_text(encoding="utf-8", errors="replace"))
else:
    print("Anomaly review report not found:", anomaly_path)

anomaly_files = sorted(REPORTS.glob("anomaly_*.csv"))
print(f"\nAnomaly CSV files found: {len(anomaly_files)}")
for path in anomaly_files:
    try:
        sample = pd.read_csv(path)
        print(f"- {path.name}: {len(sample):,} flagged/review rows")
    except Exception as exc:
        print(f"- {path.name}: could not read ({exc})")

HealthConnect Rwanda Week 6 - exploratory anomaly review
Flags use statistical/structural rules only. They are not proof of fraud, prescribing error, disease, or clinical abnormality.
Laboratory results not parseable as numeric: 5000 of 5000.
Laboratory Result values are summarized as categories only; no units or clinical reference ranges are available, so this is not a clinical abnormality assessment.
The dataset contains no Rejected insurance claims, so suspicious-claim checks cannot validate rejection detection.
Medical equipment has no status/availability field in the inspected schema; bed status is summarized separately.

- anomaly_unusual_billing.csv: 0 flagged rows / 60000 source rows. Amount outside 1.5×IQR bounds (-136173.50, 446356.50); investigate, not proof of error
- anomaly_claims_review.csv: 0 flagged rows / 42036 source rows. ClaimAmount outside 1.5×IQR bounds (-99878.11, 324206.76) or missing status; no Rejected class in source
- anomaly_lab_result_categories.csv: 0 fl

### Anomaly-review limitations

- Laboratory results are summarized as the supplied categories (`Normal` and `Abnormal`); no units or validated clinical reference ranges are available, so no clinical abnormality assessment is inferred.
- No Rejected insurance claims were available.
- Medication-frequency flags are review prompts, not evidence of prescribing errors.
- No billing rows were flagged by the selected IQR rule; this does not prove all billing is correct.
- No facility-volume or repeated-prescription rows were flagged by the current rules.
- Beds are recorded as Available in the inspected data, and medical equipment has no status/availability field.
- Statistical and structural flags are not proof of fraud, clinical abnormality, or error.


## 7. Data and target limitations

In [7]:
limitations_path = PROCESSED / "week6_target_limitations.csv"
if limitations_path.exists():
    display(pd.read_csv(limitations_path))
else:
    print("Target limitations file not found:", limitations_path)

,Task,Status,Limitation
0,Hospital admission classification,Not labelled in this script,Admission sheet appointment IDs overlap only p...
1,Insurance claim outcome (3 classes),Not trainable with current labels,Observed claim statuses are Approved and Pendi...
2,High-risk classification,Proxy target provided,Severe diagnosis is used as a transparent prox...
3,Facility clustering,Requires further validation,Facility-to-department linkage rates in the au...


## Facility clustering results

The separate facility-clustering workflow groups the 20 available FacilityIDs into three exploratory segments using appointment, doctor, nurse, department, equipment, room, and bed record counts. Beds are assigned to facilities through the supplied RoomID-to-room relationship; appointment DepartmentID is not joined to department FacilityID.

| Segment | Facilities | Mean appointment records | Mean recorded resource footprint |
|---|---:|---:|---:|
| 0 | 10 | 251.9 | 192.0 |
| 1 | 5 | 230.2 | 193.0 |
| 2 | 5 | 266.0 | 193.0 |

The recorded staffing and resource counts are identical or nearly identical across facilities, so these clusters are driven mainly by appointment-record volume. They are not facility performance ratings and should not independently drive resource allocation. See `Reports/Week6_Facility_Clustering_Report.txt` for the facility IDs and planning considerations.


## 8. Required tasks and data-blocked items

| Assignment task | Current status | Reason |
|---|---|---|
| Hospital admission classification | Blocked by target reliability | Only 586 of the 5,000 admission records link to the available appointments; unmatched appointments cannot safely be labelled as non-admissions. |
| Three-class insurance outcome | Blocked by missing class | Source statuses contain Approved and Pending, but no Rejected records. |
| High-risk patient classification | Proxy only | Severe diagnosis is used as a proxy, not a clinically validated risk label. |
| Facility clustering | Exploratory output produced | Three clusters use direct FacilityID-level counts and avoid the unreliable appointment-to-department link. Most resource counts are similar, so interpretation is limited. |
| Explainable AI (SHAP) | Implemented for proxy model | The script creates global feature importance and individual explanations for a decision tree fitted to `HighRiskProxy`; interpretation remains exploratory. LIME is not implemented. |

Do not manufacture labels to fill these gaps. The admission and three-class claim tasks require better source labels/relationships; stronger facility-level conclusions require validated facility attributes and more varied, reliable operational measures.

## 9. Responsible-use conclusion

The project has prepared modelling datasets, baseline results, patient and facility clusters, permutation feature-importance outputs, SHAP explanations for a severity-proxy model, and rule-based anomaly-review outputs. Predictive performance is weak, some required target classes are absent, and several data relationships are incomplete. The results are appropriate for an educational prototype and data-readiness review—not real patient-care, insurance, fraud, or billing decisions.

## 10. Existing project scripts and outputs

- `prepare_week6_datasets.py`
- `train_week6_models.py`
- `build_week6_model_evaluation.py`
- `build_week6_facility_clustering.py`
- `week6_explainability_anomalies.py`
- `week6_shap_explainability.py`
- `Reports/Data_Readiness_Audit.txt`
- `Reports/week6_baseline_metrics.csv`
- `Reports/Week6_Model_Comparison.csv`
- `Reports/Week6_Model_Evaluation_Report.txt`
- `Reports/Week6_Explainability_Report.txt`
- `Reports/Week6_SHAP_Explainability_Report.txt`
- `Reports/Week6_Anomaly_Review.txt`
- `Reports/Week6_Facility_Clustering_Report.txt`
- `Reports/facility_cluster_profiles.csv`

**Reproducibility note:** Run this notebook from the project root or the `Notebook` directory. The notebook reads existing outputs; it does not change the original datasets or retrain the baseline models.